In [ ]:
!pip install numpy

In [ ]:
import numpy as np

# Cargar el archivo
path = 'prueba_camilo_25_enero_2019.npz'

with np.load(path) as data:
    # Ver todas las variables disponibles (por si acaso hay más)
    print("Variables en el archivo:", data.files)
    
    # Extraer tus variables específicas
    input_data = data['input_data']
    target_data = data['target_data']
    loc_data = data['loc_data']
    input_variable_list = data['input_variable_list']
    target_variable_list = data['target_variable_list']
    lats = data['lats']
    lons = data['lons']
    
    

# Ahora puedes trabajar con ellas fuera del bloque 'with'
print(f"Forma de maxice: {input_data.shape}")

In [ ]:
target_data

In [ ]:
# 1. Parámetros de la grilla (fijos)
grid_h, grid_w = 299, 299
num_vars = 3  # maxice, intice, etc.


# Cargar el archivo
path = 'prueba_camilo_25_enero_2019.npz'

with np.load(path) as data:
    
# 2. Cargar datos (Simulación de tu carga actual)
# input_data.shape -> (2682030, 3)
    input_raw = data['input_data']
# 1. Cargar el target raw
# shape actual: (2682030, 1)
    target_raw = data['target_data']

# 3. Calcular dinámicamente el número de pasos de tiempo
# El total de filas (2682030) debe ser igual a: (Tiempos * Grid_H * Grid_W)
total_points_per_var = input_raw.shape[0]
pixels_per_slice = grid_h * grid_w

if total_points_per_var % pixels_per_slice != 0:
    raise ValueError("El tamaño total no coincide con una grilla de 299x299.")

num_times = total_points_per_var // pixels_per_slice
print(f"Detectados {num_times} pasos de tiempo.")

# 4. Reshape para la CNN
# Queremos pasar de (Puntos_Totales, Canales) -> (Tiempos, Filas, Columnas, Canales)
input_cnn = input_raw.reshape((num_times, grid_h, grid_w, num_vars))

print(f"Shape final para la CNN: {input_cnn.shape}")
# Resultado esperado: (30, 299, 299, 3)

In [ ]:
# Definir cuántos píxeles quieres eliminar de los bordes
#border_size = 5 

# Aplicar el recorte (slicing)
# Estructura: [tiempo, filas, columnas, variables]
#input_cleaned = input_cnn[:, border_size:-border_size, border_size:-border_size, :]

#print(f"Shape original: {input_cnn.shape}")
#print(f"Shape después de limpiar bordes: {input_cleaned.shape}")
# Si border_size es 5, la nueva forma será (30, 289, 289, 3)

In [ ]:
# 2. Reshape del target
# Usamos las mismas variables grid_h, grid_w y num_times calculadas antes
# para asegurar que siempre coincidan.
target_cnn = target_raw.reshape((num_times, grid_h, grid_w, 1))

print(f"Shape final del Target para la CNN: {target_cnn.shape}")
# Resultado: (30, 299, 299, 1)

In [ ]:
# Verificar que el primer "píxel" del input y target en el tiempo 0 coincida con el dato original
assert input_cnn[0, 0, 0, 0] == input_raw[0, 0]
assert target_cnn[0, 0, 0, 0] == target_raw[0, 0]

print("¡Reshape correcto! La integridad de los datos se mantiene.")

In [ ]:
!pip install tensorflow

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, UpSampling2D, concatenate, ZeroPadding2D, Cropping2D
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D


X_full = input_cnn

Y_full = target_cnn

# ==========================================
# 2. SPLIT TEMPORAL (Evitar Data Leakage)
# ==========================================
print("2. Separando datos temporalmente en Train y Validation...")

# Asumimos que X_full y Y_full tienen shapes (tiempos, 299, 299, variables/canales)
num_times = X_full.shape[0]

# Usamos el 80% cronológico para entrenar y el 20% final para validar
split_idx = int(num_times * 0.8)

X_train = X_full[:split_idx]
y_train = Y_full[:split_idx]

X_val = X_full[split_idx:]
y_val = Y_full[split_idx:]

print(f"Tiempos para Entrenamiento: {X_train.shape[0]}")
print(f"Tiempos para Validación: {X_val.shape[0]}")

# ==========================================
# 3. SHUFFLE (Solo al set de entrenamiento)
# ==========================================
print("\n3. Aplicando Shuffle al set de entrenamiento...")
np.random.seed(42) # Semilla para reproducibilidad
indices_train = np.arange(X_train.shape[0])
np.random.shuffle(indices_train)

X_train = X_train[indices_train]
y_train = y_train[indices_train]


#### NORMALIZANDO!!!

print("Normalizando las variables de entrada (Min-Max Scaling)...")

# 1. Calculamos el Mínimo y Máximo de CADA variable (canal) 
# axis=(0, 1, 2) significa que busca el max/min a través de todos los tiempos y todos los píxeles, 
# dejando un valor distinto para cada una de tus 3 variables.
min_vals = np.min(X_train, axis=(0, 1, 2), keepdims=True)
max_vals = np.max(X_train, axis=(0, 1, 2), keepdims=True)

# El '+ 1e-8' es un truquito matemático para evitar que el código explote (división por cero) 
# si alguna variable llegara a ser toda ceros.
rango = max_vals - min_vals + 1e-8

# 2. Normalizamos el set de ENTRENAMIENTO
X_train_norm = (X_train - min_vals) / rango

# 3. Normalizamos el set de VALIDACIÓN (Usando los min y max del entrenamiento)
X_val_norm = (X_val - min_vals) / rango

print(f"Rango X_train original: min {X_train.min():.4f}, max {X_train.max():.4f}")
print(f"Rango X_train normalizado: min {X_train_norm.min():.4f}, max {X_train_norm.max():.4f}")


## aplicamos transformada logaritmica al target: 

# 1. Transformamos el target (Densidad de Rayos) a escala logarítmica
# Usamos log1p (log(1 + x)) porque tenemos muchos ceros, y el log(0) da error.
y_train_log = np.log1p(y_train)
y_val_log = np.log1p(y_val)

In [ ]:
def build_baseline_fcn(input_shape=(299, 299, 3)):
    model = Sequential([
        # Extraemos características básicas
        Conv2D(32, (3, 3), activation='relu', padding='same', input_shape=input_shape),
        Conv2D(64, (3, 3), activation='relu', padding='same'),
        Conv2D(32, (3, 3), activation='relu', padding='same'),
        
        # Capa de salida: 1 solo canal (lightdens)
        # Usamos kernel (1,1) para combinar los canales finales en uno solo
        # Usamos 'relu' o 'linear' porque la densidad de rayos NO puede ser negativa
        Conv2D(1, (1, 1), activation='linear', padding='same') 
    ])
    return model

model = build_baseline_fcn()
model.summary()

# Compilación
# mse (Mean Squared Error) suele ser el estándar para regresión
model.compile(optimizer='adam', loss='mse', metrics=['mae'])

# 5. ENTRENAMIENTO DEL MODELO
# ==========================================
print("\n5. Iniciando Entrenamiento...")

# EarlyStopping detiene el entrenamiento si el modelo deja de mejorar, evitando overfitting
early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

history = model.fit(
    X_train_norm, y_train_log,
    validation_data=(X_val_norm, y_val_log),
    epochs=50, # Número máximo de pasadas por los datos
    batch_size=4, # Dependerá de cuánta VRAM tenga tu GPU. Si da error de memoria (OOM), bájalo a 2 o 1.
    callbacks=[early_stop],
    verbose=1
)

print("\n¡Entrenamiento finalizado exitosamente!")

In [ ]:
!pip install matplotlib

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# 1. Elegimos un índice al azar del set de VALIDACIÓN
# (Queremos ver cómo le va con datos que nunca vio en el entrenamiento)
idx = 3 # Podés cambiar este número para ver otros instantes (de 0 a X_val.shape[0]-1)

# Extraemos la muestra y le agregamos la dimensión del batch (1, 299, 299, 3)
muestra_x = np.expand_dims(X_val_norm[idx], axis=0)
muestra_y_real = y_val_log[idx]

# Hacemos la predicción (que saldrá en escala logarítmica)
prediccion_log = model.predict(muestra_x)

# 🚀 Aplicamos la inversa para volver a la escala original de rayos
prediccion_real_scale = np.expm1(prediccion_log)

# Acordate de quitar la dimensión del batch para graficar
prediccion_grafico = prediccion_real_scale[0] 

# (Y graficás 'prediccion_grafico' como venías haciendo)

# 3. Graficamos
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Plot 1: Una de las variables de entrada (ej. maxice)
# Nota: Si maxice es el índice 0, usamos [:, :, 0]
im0 = axes[0].imshow(X_val_norm[idx, :, :, 0], cmap='Blues', origin='lower')
axes[0].set_title('Entrada: Maxice')
fig.colorbar(im0, ax=axes[0])

# Plot 2: El Target Real (Lo que realmente pasó)
im1 = axes[1].imshow(np.expm1(muestra_y_real[:, :, 0]), cmap='magma', origin='lower')
axes[1].set_title('Real: Densidad de Rayos')
fig.colorbar(im1, ax=axes[1])

# Plot 3: La Predicción de tu Modelo
im2 = axes[2].imshow(prediccion_grafico[:, :, 0], cmap='magma', origin='lower')
axes[2].set_title('Predicción: Baseline')
fig.colorbar(im2, ax=axes[2])

plt.tight_layout()
plt.show()

In [ ]:
# ==========================================
# 4. CONSTRUCCIÓN DEL MODELO U-NET (Adaptado para 299x299)
# ==========================================
print("\n4. Construyendo arquitectura U-Net...")

def build_unet(input_shape=(299, 299, 3)):
    inputs = Input(input_shape)

    # --- Ajuste interno de dimensiones ---
    # Rellenamos con ceros para llevar la imagen de 299x299 a 304x304 (divisible por 16)
    # Agregamos 2 pixeles arriba, 3 abajo, 2 a la izquierda, 3 a la derecha
    padded_inputs = ZeroPadding2D(padding=((2, 3), (2, 3)))(inputs)

    # --- Bajada (Encoder) ---
    c1 = Conv2D(32, (3, 3), activation='relu', padding='same')(padded_inputs)
    c1 = Conv2D(32, (3, 3), activation='relu', padding='same')(c1)
    p1 = MaxPooling2D((2, 2))(c1) # Baja a 152x152

    c2 = Conv2D(64, (3, 3), activation='relu', padding='same')(p1)
    c2 = Conv2D(64, (3, 3), activation='relu', padding='same')(c2)
    p2 = MaxPooling2D((2, 2))(c2) # Baja a 76x76
    
    # --- Cuello de botella (Bottleneck) ---
    c3 = Conv2D(128, (3, 3), activation='relu', padding='same')(p2)
    c3 = Conv2D(128, (3, 3), activation='relu', padding='same')(c3)

    # --- Subida (Decoder) ---
    u4 = UpSampling2D((2, 2))(c3) # Sube a 152x152
    u4 = concatenate([u4, c2]) 
    c4 = Conv2D(64, (3, 3), activation='relu', padding='same')(u4)
    c4 = Conv2D(64, (3, 3), activation='relu', padding='same')(c4)

    u5 = UpSampling2D((2, 2))(c4) # Sube a 304x304
    u5 = concatenate([u5, c1]) 
    c5 = Conv2D(32, (3, 3), activation='relu', padding='same')(u5)
    c5 = Conv2D(32, (3, 3), activation='relu', padding='same')(c5)

    # --- Capa de Salida y Reajuste ---
    # 1 solo canal para la densidad de rayos
    outputs_padded = Conv2D(1, (1, 1), activation='linear')(c5)
    
    # Recortamos exactamente lo que rellenamos al principio para volver a 299x299
    outputs = Cropping2D(cropping=((2, 3), (2, 3)))(outputs_padded)

    model = Model(inputs=[inputs], outputs=[outputs])
    return model

unet_model = build_unet()

# Compilación
unet_model.compile(optimizer='adam', loss='mse', metrics=['mae'])

# ==========================================
# 5. ENTRENAMIENTO DEL MODELO
# ==========================================
print("\n5. Iniciando Entrenamiento...")

# EarlyStopping detiene el entrenamiento si el modelo deja de mejorar
early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

history = unet_model.fit(
    X_train_norm, y_train_log,
    validation_data=(X_val_norm, y_val_log),
    epochs=50, 
    batch_size=4, # Si tu GPU se queda sin memoria, bajalo a 2 o 1
    callbacks=[early_stop],
    verbose=1
)

print("\n¡Entrenamiento finalizado exitosamente!")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# 1. Elegimos un índice al azar del set de VALIDACIÓN
# (Queremos ver cómo le va con datos que nunca vio en el entrenamiento)
idx = 3 # Podés cambiar este número para ver otros instantes (de 0 a X_val.shape[0]-1)

# Extraemos la muestra y le agregamos la dimensión del batch (1, 299, 299, 3)
muestra_x = np.expand_dims(X_val_norm[idx], axis=0)
muestra_y_real = y_val_log[idx]

# Hacemos la predicción (que saldrá en escala logarítmica)
prediccion_log = unet_model.predict(muestra_x)

# 🚀 Aplicamos la inversa para volver a la escala original de rayos
prediccion_real_scale = np.expm1(prediccion_log)

# Acordate de quitar la dimensión del batch para graficar
prediccion_grafico = prediccion_real_scale[0] 

# (Y graficás 'prediccion_grafico' como venías haciendo)

# 3. Graficamos
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Plot 1: Una de las variables de entrada (ej. maxice)
# Nota: Si maxice es el índice 0, usamos [:, :, 0]
im0 = axes[0].imshow(X_val_norm[idx, :, :, 0], cmap='Blues', origin='lower')
axes[0].set_title('Entrada: Maxice')
fig.colorbar(im0, ax=axes[0])

# Plot 2: El Target Real (Lo que realmente pasó)
im1 = axes[1].imshow(np.expm1(muestra_y_real[:, :, 0]), cmap='magma', origin='lower')
axes[1].set_title('Real: Densidad de Rayos')
fig.colorbar(im1, ax=axes[1])

# Plot 3: La Predicción de tu Modelo
im2 = axes[2].imshow(prediccion_grafico[:, :, 0], cmap='magma', origin='lower')
axes[2].set_title('Predicción: UNET')
fig.colorbar(im2, ax=axes[2])

plt.tight_layout()
plt.show()

In [ ]:
### modificando la funcion de perdida ####

### La nueva Función de Pérdida Ponderada
### Como el logaritmo achicó mucho las diferencias, le vamos a decir a la red que errarle a un píxel con rayos es 50 veces peor que errarle a un píxel despejado.


import tensorflow.keras.backend as K
import tensorflow as tf

def weighted_mse_log(y_true, y_pred):
    # Si hay rayos, el peso es 50. Si es cielo despejado, es 1.
    pesos = tf.where(y_true > 0.0, 50.0, 1.0)
    error_cuadrado = K.square(y_true - y_pred)
    return K.mean(error_cuadrado * pesos)

# Compilamos con esta nueva función
unet_model.compile(optimizer='adam', loss=weighted_mse_log, metrics=['mae'])


history_unet = unet_model.fit(
    X_train_norm, y_train, 
    validation_data=(X_val_norm, y_val),
    epochs=50, 
    batch_size=4,
    callbacks=[early_stop],
    verbose=1
)



In [ ]:

# 1. Elegimos un índice al azar del set de VALIDACIÓN
# (Queremos ver cómo le va con datos que nunca vio en el entrenamiento)
idx = 3 # Podés cambiar este número para ver otros instantes (de 0 a X_val.shape[0]-1)

# Extraemos la muestra y le agregamos la dimensión del batch (1, 299, 299, 3)
muestra_x = np.expand_dims(X_val_norm[idx], axis=0)
#muestra_y_real = y_val_log[idx]
muestra_y_real = y_val[idx]

# Hacemos la predicción (que saldrá en escala logarítmica)
prediccion_log = unet_model.predict(muestra_x)

# 🚀 Aplicamos la inversa para volver a la escala original de rayos
prediccion_real_scale = np.expm1(prediccion_log)

# Acordate de quitar la dimensión del batch para graficar
#prediccion_grafico = prediccion_real_scale[0] 
prediccion_grafico = prediccion_log[0] 

# (Y graficás 'prediccion_grafico' como venías haciendo)

# 3. Graficamos
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Plot 1: Una de las variables de entrada (ej. maxice)
# Nota: Si maxice es el índice 0, usamos [:, :, 0]
im0 = axes[0].imshow(X_val_norm[idx, :, :, 0], cmap='Blues', origin='lower')
axes[0].set_title('Entrada: Maxice')
fig.colorbar(im0, ax=axes[0])

# Plot 2: El Target Real (Lo que realmente pasó)
#im1 = axes[1].imshow(np.expm1(muestra_y_real[:, :, 0]), cmap='magma', origin='lower')
im1 = axes[1].imshow((muestra_y_real[:, :, 0]), cmap='magma', origin='lower')
axes[1].set_title('Real: Densidad de Rayos')
fig.colorbar(im1, ax=axes[1])

# Plot 3: La Predicción de tu Modelo
im2 = axes[2].imshow(prediccion_grafico[:, :, 0], cmap='magma', origin='lower')
axes[2].set_title('Predicción: UNET + loss super mala + sin log')
fig.colorbar(im2, ax=axes[2])

plt.tight_layout()
plt.show()